In [1]:
# ============================================================
# FULL HUP RETRAINING SCRIPT
# CNN + Transformer backbone
# + Masked Attention Pooling
# + Subject-Adversarial Branch
# + Flexible threshold selection
# + 0.5 fallback threshold
# + Fixed per-subject retention during training
# + Saves best model for every fold / reserved run
# ============================================================

from __future__ import annotations

import copy
import json
import math
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from torch.autograd import Function
from torch.utils.data import DataLoader, Dataset

C:\Users\ajars\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
# ============================================================
# CONFIG
# ============================================================

PREPROCESSED_ROOT = Path(r"D:\HUP_processed_ver2")
EXPERIMENT_ROOT = Path(r"D:\hup_all_subjects_adv_transformer")
EXPERIMENT_ROOT.mkdir(parents=True, exist_ok=True)

MODELS_DIR = EXPERIMENT_ROOT / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# empty = use all subjects found in PREPROCESSED_ROOT
SUBJECT_FILTER: List[str] = []

# reserved subjects for LLM evaluation
LLM_COHORT_SUBJECTS: List[str] = [
    "sub-HUP126",
    "sub-HUP134",
    "sub-HUP140",
    "sub-HUP146",
    "sub-HUP164",
]

# Optional manual override for reserved runs.
# Example:
# MANUAL_LLM_HOLDOUTS = {
#     "sub-HUP146": {
#         "ictal": "sub-HUP146_ses-presurgery_task-ictal_acq-seeg_run-03",
#     }
# }
MANUAL_LLM_HOLDOUTS: Dict[str, Dict[str, str]] = {}

SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# training
BATCH_SIZE = 32
NUM_EPOCHS = 35
LR = 3e-4
WEIGHT_DECAY = 3e-4
EARLY_STOPPING_PATIENCE = 5

# model
DROPOUT = 0.30
CNN_HIDDEN = 32
EMBED_DIM = 64
NHEAD = 2
NUM_LAYERS = 1
FF_MULT = 4

# loss
FOCAL_GAMMA = 2.0
FOCAL_ALPHA = 0.5
ADV_WEIGHT = 0.2
GRL_LAMBDA = 1.0

# smoothing + thresholding
SMOOTHING_KERNEL = 5
THRESH_GRID = [round(x, 2) for x in np.arange(0.10, 0.91, 0.05)]
FALLBACK_THRESHOLD = 0.50

# subject retention during training
MAX_TRAIN_ICTAL_PER_SUBJECT = 100
MAX_TRAIN_NONICTAL_PER_SUBJECT = 100

# optional minimum
MIN_ICTAL_WINDOWS_FOR_TRAIN = 1

In [3]:
# ============================================================
# SEED
# ============================================================

def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

In [4]:
# ============================================================
# METRICS / UTILS
# ============================================================

def moving_average(x: np.ndarray, k: int) -> np.ndarray:
    if k <= 1 or len(x) == 0:
        return x.copy()
    pad = k // 2
    xpad = np.pad(x, (pad, pad), mode="edge")
    kernel = np.ones(k, dtype=np.float32) / float(k)
    return np.convolve(xpad, kernel, mode="valid")


def safe_roc_auc(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    if len(np.unique(y_true)) < 2:
        return float("nan")
    return float(roc_auc_score(y_true, y_prob))


def safe_auprc(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    if len(np.unique(y_true)) < 2:
        return float("nan")
    return float(average_precision_score(y_true, y_prob))


def compute_metrics(y_true: np.ndarray, y_prob: np.ndarray, threshold: float) -> Dict[str, float]:
    y_pred = (y_prob >= threshold).astype(np.int64)

    acc = accuracy_score(y_true, y_pred)
    bal_acc = balanced_accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    auroc = safe_roc_auc(y_true, y_prob)
    auprc = safe_auprc(y_true, y_prob)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    return {
        "acc": float(acc),
        "balanced_acc": float(bal_acc),
        "precision": float(prec),
        "recall": float(rec),
        "f1": float(f1),
        "auroc": float(auroc) if not math.isnan(auroc) else np.nan,
        "auprc": float(auprc) if not math.isnan(auprc) else np.nan,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


def select_best_threshold(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    thresh_grid: List[float],
    fallback_threshold: float = 0.5,
) -> float:
    # degenerate validation
    if len(np.unique(y_true)) < 2:
        return fallback_threshold

    best_thr = fallback_threshold
    best_key = (-1.0, -1.0, -1.0)  # balanced_acc, f1, precision

    for thr in thresh_grid:
        m = compute_metrics(y_true, y_prob, thr)
        key = (m["balanced_acc"], m["f1"], m["precision"])
        if key > best_key:
            best_key = key
            best_thr = thr

    return float(best_thr)

In [5]:
# ============================================================
# RUN SCANNING
# ============================================================

@dataclass
class RunRecord:
    subject_id: str
    run_stem: str
    npz_path: Path
    meta_path: Path
    task: str
    acquisition: str
    n_windows: int
    n_ictal: int
    n_nonictal: int


def parse_stem(stem: str) -> Dict[str, str]:
    # sub-HUP146_ses-presurgery_task-ictal_acq-seeg_run-03
    parts = stem.split("_")
    return {
        "subject_id": parts[0],
        "session": parts[1],
        "task": parts[2].replace("task-", ""),
        "acquisition": parts[3].replace("acq-", ""),
        "run": parts[4].replace("run-", ""),
    }


def scan_runs(preprocessed_root: Path) -> List[RunRecord]:
    runs: List[RunRecord] = []
    for npz_path in sorted(preprocessed_root.glob("*.npz")):
        stem = npz_path.stem
        meta_path = preprocessed_root / f"{stem}_meta.json"
        if not meta_path.exists():
            continue

        info = parse_stem(stem)
        with open(meta_path, "r", encoding="utf-8") as f:
            meta = json.load(f)

        counts = meta.get("class_counts", {})
        runs.append(
            RunRecord(
                subject_id=info["subject_id"],
                run_stem=stem,
                npz_path=npz_path,
                meta_path=meta_path,
                task=info["task"],
                acquisition=info["acquisition"],
                n_windows=int(counts.get("n_total", 0)),
                n_ictal=int(counts.get("n_ictal", 0)),
                n_nonictal=int(counts.get("n_nonictal", 0)),
            )
        )
    return runs


def filter_runs(runs: List[RunRecord]) -> List[RunRecord]:
    out = []
    for r in runs:
        if SUBJECT_FILTER and r.subject_id not in SUBJECT_FILTER:
            continue
        if r.task != "ictal":
            continue
        out.append(r)
    return out


def build_subject_run_map(runs: List[RunRecord]) -> Dict[str, List[RunRecord]]:
    out: Dict[str, List[RunRecord]] = {}
    for r in runs:
        out.setdefault(r.subject_id, []).append(r)
    for s in out:
        out[s] = sorted(out[s], key=lambda x: x.run_stem)
    return out


In [6]:
# ============================================================
# DATA LOADING
# ============================================================

def load_npz(npz_path: Path) -> Dict[str, np.ndarray]:
    d = np.load(npz_path)
    return {
        "X": d["X"].astype(np.float32),           # (N, C, T)
        "y": d["y"].astype(np.int64),             # eval labels
        "y_train": d["y_train"].astype(np.int64), # train labels (-1 ignore)
        "mask": d["mask"].astype(bool),           # (N, C)
        "t_bounds": d["t_bounds"].astype(np.float32),
    }


def concatenate_runs_with_subject_idx(run_records: List[RunRecord], subject_to_idx: Dict[str, int]):
    blocks = []
    subj_idx_list = []

    for r in run_records:
        b = load_npz(r.npz_path)
        blocks.append(b)
        subj_idx_list.append(np.full(len(b["y"]), subject_to_idx[r.subject_id], dtype=np.int64))

    X = np.concatenate([b["X"] for b in blocks], axis=0)
    y = np.concatenate([b["y"] for b in blocks], axis=0)
    y_train = np.concatenate([b["y_train"] for b in blocks], axis=0)
    mask = np.concatenate([b["mask"] for b in blocks], axis=0)
    t_bounds = np.concatenate([b["t_bounds"] for b in blocks], axis=0)
    subject_idx = np.concatenate(subj_idx_list, axis=0)

    run_ids = []
    for r, b in zip(run_records, blocks):
        run_ids.extend([r.run_stem] * len(b["y"]))
    run_ids = np.asarray(run_ids)

    return {
        "X": X,
        "y": y,
        "y_train": y_train,
        "mask": mask,
        "t_bounds": t_bounds,
        "subject_idx": subject_idx,
        "run_ids": run_ids,
    }


def retain_subject_quota_adv(
    X: np.ndarray,
    y_eval: np.ndarray,
    y_train: np.ndarray,
    mask: np.ndarray,
    t_bounds: np.ndarray,
    subject_idx: np.ndarray,
    run_ids: np.ndarray,
    max_ictal: int = 100,
    max_nonictal: int = 100,
    seed: int = 42,
):
    """
    Apply fixed per-subject retention.
    Since this function is called inside a subject-specific fold,
    it effectively enforces fixed quota for that subject's training pool.
    """
    rng = np.random.default_rng(seed)

    valid_idx = np.where(y_train != -1)[0]
    ictal_idx = valid_idx[y_train[valid_idx] == 1]
    nonictal_idx = valid_idx[y_train[valid_idx] == 0]

    if len(ictal_idx) > max_ictal:
        ictal_idx = np.sort(rng.choice(ictal_idx, size=max_ictal, replace=False))
    if len(nonictal_idx) > max_nonictal:
        nonictal_idx = np.sort(rng.choice(nonictal_idx, size=max_nonictal, replace=False))

    keep_idx = np.sort(np.concatenate([ictal_idx, nonictal_idx]))

    return (
        X[keep_idx],
        y_eval[keep_idx],
        y_train[keep_idx],
        mask[keep_idx],
        t_bounds[keep_idx],
        subject_idx[keep_idx],
        run_ids[keep_idx],
    )


In [7]:
# ============================================================
# DATASETS
# ============================================================

class EEGDatasetAdv(Dataset):
    def __init__(
        self,
        X: np.ndarray,
        y_train: np.ndarray,
        mask: np.ndarray,
        subject_idx: np.ndarray,
    ):
        keep = y_train != -1
        self.X = torch.from_numpy(X[keep]).float()
        self.y = torch.from_numpy(y_train[keep]).float()
        self.mask = torch.from_numpy(mask[keep]).bool()
        self.subject_idx = torch.from_numpy(subject_idx[keep]).long()

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.mask[idx], self.y[idx], self.subject_idx[idx]


class EEGEvalDataset(Dataset):
    def __init__(
        self,
        X: np.ndarray,
        y_eval: np.ndarray,
        mask: np.ndarray,
    ):
        self.X = torch.from_numpy(X).float()
        self.y = torch.from_numpy(y_eval).float()
        self.mask = torch.from_numpy(mask).bool()

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.mask[idx], self.y[idx]


In [8]:
# ============================================================
# MODEL
# ============================================================

class GradientReversalFn(Function):
    @staticmethod
    def forward(ctx, x, lambd):
        ctx.lambd = lambd
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.lambd * grad_output, None


class GradientReversal(nn.Module):
    def __init__(self, lambd: float = 1.0):
        super().__init__()
        self.lambd = lambd

    def forward(self, x):
        return GradientReversalFn.apply(x, self.lambd)


class ConvFeatureExtractor(nn.Module):
    def __init__(self, cnn_hidden: int = 32, embed_dim: int = 64, dropout: float = 0.3):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(1, cnn_hidden, kernel_size=7, stride=2, padding=3),
            nn.BatchNorm1d(cnn_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Conv1d(cnn_hidden, cnn_hidden * 2, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm1d(cnn_hidden * 2),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Conv1d(cnn_hidden * 2, embed_dim, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm1d(embed_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.pool = nn.AdaptiveAvgPool1d(1)

    def forward(self, x):
        # x: (B*C, 1, T)
        z = self.conv(x)
        z = self.pool(z).squeeze(-1)
        return z


class MaskedAttentionPooling(nn.Module):
    def __init__(self, embed_dim: int):
        super().__init__()
        self.score = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.Tanh(),
            nn.Linear(embed_dim, 1),
        )

    def forward(self, z, ch_mask):
        scores = self.score(z).squeeze(-1)      # (B, C)
        scores = scores.masked_fill(~ch_mask, float("-inf"))
        attn = torch.softmax(scores, dim=1)
        attn = torch.nan_to_num(attn, nan=0.0, posinf=0.0, neginf=0.0)
        pooled = torch.sum(z * attn.unsqueeze(-1), dim=1)
        return pooled, attn


class SeizureCNNTransformerAdv(nn.Module):
    def __init__(
        self,
        embed_dim: int = 64,
        nhead: int = 2,
        num_layers: int = 1,
        ff_mult: int = 4,
        dropout: float = 0.3,
        num_subjects: int = 10,
        grl_lambda: float = 1.0,
    ):
        super().__init__()

        self.feature = ConvFeatureExtractor(
            cnn_hidden=CNN_HIDDEN,
            embed_dim=embed_dim,
            dropout=dropout,
        )

        enc_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=nhead,
            dim_feedforward=embed_dim * ff_mult,
            dropout=dropout,
            batch_first=True,
            norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.pool = MaskedAttentionPooling(embed_dim)

        self.seizure_head = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Dropout(dropout),
            nn.Linear(embed_dim, 1),
        )

        self.grl = GradientReversal(lambd=grl_lambda)
        self.subject_head = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Dropout(dropout),
            nn.Linear(embed_dim, embed_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim, num_subjects),
        )

    def forward(self, x, ch_mask):
        B, C, T = x.shape

        z = x.reshape(B * C, 1, T)
        z = self.feature(z)
        z = z.reshape(B, C, -1)

        key_padding_mask = ~ch_mask
        z = self.transformer(z, src_key_padding_mask=key_padding_mask)

        pooled, attn = self.pool(z, ch_mask)

        seizure_logits = self.seizure_head(pooled).squeeze(-1)

        adv_feat = self.grl(pooled)
        subject_logits = self.subject_head(adv_feat)

        return {
            "seizure_logits": seizure_logits,
            "subject_logits": subject_logits,
            "channel_attn": attn,
            "pooled": pooled,
        }


In [9]:
# ============================================================
# LOSS
# ============================================================

class BinaryFocalLoss(nn.Module):
    def __init__(self, alpha: float = 0.5, gamma: float = 2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        probs = torch.sigmoid(logits)
        pt = torch.where(targets == 1, probs, 1 - probs)
        alpha_t = torch.where(targets == 1, self.alpha, 1 - self.alpha)
        loss = alpha_t * ((1 - pt) ** self.gamma) * bce
        return loss.mean()


In [10]:
# ============================================================
# PREDICT / TRAIN
# ============================================================

@torch.no_grad()
def predict_probs_adv(model: nn.Module, loader: DataLoader, device: str):
    model.eval()
    all_probs, all_y = [], []

    for X, mask, y in loader:
        X = X.to(device)
        mask = mask.to(device)

        out = model(X, mask)
        probs = torch.sigmoid(out["seizure_logits"]).cpu().numpy()

        all_probs.append(probs)
        all_y.append(y.numpy())

    return np.concatenate(all_probs), np.concatenate(all_y)


def train_one_fold_adv(
    train_data: Dict[str, np.ndarray],
    val_data: Dict[str, np.ndarray],
    test_data: Dict[str, np.ndarray],
    subject_to_idx: Dict[str, int],
    device: str = DEVICE,
    adv_weight: float = ADV_WEIGHT,
    grl_lambda: float = GRL_LAMBDA,
):
    Xtr, ytr_eval, ytr_train, mtr, tbtr, subjtr, ridtr = retain_subject_quota_adv(
        train_data["X"], train_data["y"], train_data["y_train"],
        train_data["mask"], train_data["t_bounds"],
        train_data["subject_idx"], train_data["run_ids"],
        max_ictal=MAX_TRAIN_ICTAL_PER_SUBJECT,
        max_nonictal=MAX_TRAIN_NONICTAL_PER_SUBJECT,
        seed=SEED,
    )

    train_ds = EEGDatasetAdv(Xtr, ytr_train, mtr, subjtr)
    val_ds = EEGEvalDataset(val_data["X"], val_data["y"], val_data["mask"])
    test_ds = EEGEvalDataset(test_data["X"], test_data["y"], test_data["mask"])

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

    model = SeizureCNNTransformerAdv(
        embed_dim=EMBED_DIM,
        nhead=NHEAD,
        num_layers=NUM_LAYERS,
        ff_mult=FF_MULT,
        dropout=DROPOUT,
        num_subjects=len(subject_to_idx),
        grl_lambda=grl_lambda,
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    seizure_criterion = BinaryFocalLoss(alpha=FOCAL_ALPHA, gamma=FOCAL_GAMMA)
    subject_criterion = nn.CrossEntropyLoss()

    best_state = None
    best_score = -1.0
    patience = 0
    history = []

    for epoch in range(1, NUM_EPOCHS + 1):
        model.train()
        running_loss = 0.0

        for X, mask, y, subj_idx in train_loader:
            X = X.to(device)
            mask = mask.to(device)
            y = y.to(device)
            subj_idx = subj_idx.to(device)

            optimizer.zero_grad()

            out = model(X, mask)
            seizure_logits = out["seizure_logits"]
            subject_logits = out["subject_logits"]

            seizure_loss = seizure_criterion(seizure_logits, y)
            subject_loss = subject_criterion(subject_logits, subj_idx)
            loss = seizure_loss + adv_weight * subject_loss

            loss.backward()
            optimizer.step()

            running_loss += loss.item() * len(y)

        train_loss = running_loss / max(1, len(train_ds))

        val_prob, val_y = predict_probs_adv(model, val_loader, device)
        val_prob_s = moving_average(val_prob, SMOOTHING_KERNEL)

        val_thr = select_best_threshold(
            val_y.astype(np.int64),
            val_prob_s.astype(np.float32),
            THRESH_GRID,
            fallback_threshold=FALLBACK_THRESHOLD,
        )
        val_metrics = compute_metrics(val_y.astype(np.int64), val_prob_s, val_thr)

        score = val_metrics["balanced_acc"]
        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "val_threshold": val_thr,
            **{f"val_{k}": v for k, v in val_metrics.items()},
        })

        if score > best_score:
            best_score = score
            best_state = copy.deepcopy(model.state_dict())
            patience = 0
        else:
            patience += 1
            if patience >= EARLY_STOPPING_PATIENCE:
                break

    if best_state is None:
        best_state = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)

    val_prob, val_y = predict_probs_adv(model, val_loader, device)
    val_prob_s = moving_average(val_prob, SMOOTHING_KERNEL)
    best_thr = select_best_threshold(
        val_y.astype(np.int64),
        val_prob_s.astype(np.float32),
        THRESH_GRID,
        fallback_threshold=FALLBACK_THRESHOLD,
    )

    test_prob, test_y = predict_probs_adv(model, test_loader, device)
    test_prob_s = moving_average(test_prob, SMOOTHING_KERNEL)
    test_metrics = compute_metrics(test_y.astype(np.int64), test_prob_s, best_thr)

    return {
        "model_state": best_state,
        "history": history,
        "best_threshold": best_thr,
        "test_prob": test_prob_s,
        "test_y": test_y,
        "test_metrics": test_metrics,
        "train_n_after_retention": len(train_ds),
    }



In [11]:
# ============================================================
# SAVE ARTIFACTS
# ============================================================

def save_fold_artifacts(
    subject_id: str,
    fold_name: str,
    result: Dict,
    model_config: Dict,
    train_runs: List[str],
    val_runs: List[str],
    test_runs: List[str],
):
    out_dir = MODELS_DIR / subject_id / fold_name
    out_dir.mkdir(parents=True, exist_ok=True)

    torch.save(result["model_state"], out_dir / "best_model.pt")

    meta = {
        "subject_id": subject_id,
        "fold_name": fold_name,
        "best_threshold": float(result["best_threshold"]),
        "train_runs": train_runs,
        "val_runs": val_runs,
        "test_runs": test_runs,
        "train_n_after_retention": int(result["train_n_after_retention"]),
        "model_config": model_config,
        "test_metrics": result["test_metrics"],
    }

    with open(out_dir / "meta.json", "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2)

    pd.DataFrame(result["history"]).to_csv(out_dir / "history.csv", index=False)

    pd.DataFrame({
        "y_true": result["test_y"].astype(int),
        "y_prob": result["test_prob"].astype(float),
    }).to_csv(out_dir / "test_predictions.csv", index=False)

    return out_dir



In [12]:
# ============================================================
# SPLIT HELPERS
# ============================================================

def choose_validation_run(train_runs: List[RunRecord], held_out_run_stem: str) -> str:
    candidates = [r for r in train_runs if r.run_stem != held_out_run_stem]
    candidates = sorted(candidates, key=lambda r: (r.n_windows, r.run_stem), reverse=True)
    return candidates[0].run_stem


def find_reserved_runs(subject_runs: List[RunRecord], subject_id: str) -> Tuple[Optional[str], Optional[str]]:
    if subject_id in MANUAL_LLM_HOLDOUTS:
        rr = MANUAL_LLM_HOLDOUTS[subject_id]
        return rr.get("ictal"), rr.get("nonictal")

    ictal_runs = [r for r in subject_runs if r.task == "ictal"]
    if not ictal_runs:
        return None, None
    ictal_runs = sorted(ictal_runs, key=lambda r: (r.n_windows, r.run_stem), reverse=True)
    return ictal_runs[0].run_stem, None



In [13]:
# ============================================================
# MAIN
# ============================================================

def main():
    all_runs = filter_runs(scan_runs(PREPROCESSED_ROOT))
    subject_map = build_subject_run_map(all_runs)

    # global subject index mapping for adversarial branch
    all_subjects = sorted(subject_map.keys())
    subject_to_idx = {s: i for i, s in enumerate(all_subjects)}

    model_config = {
        "embed_dim": EMBED_DIM,
        "nhead": NHEAD,
        "num_layers": NUM_LAYERS,
        "ff_mult": FF_MULT,
        "dropout": DROPOUT,
        "adv_weight": ADV_WEIGHT,
        "grl_lambda": GRL_LAMBDA,
        "threshold_grid": THRESH_GRID,
        "fallback_threshold": FALLBACK_THRESHOLD,
        "max_train_ictal_per_subject": MAX_TRAIN_ICTAL_PER_SUBJECT,
        "max_train_nonictal_per_subject": MAX_TRAIN_NONICTAL_PER_SUBJECT,
    }

    internal_rows = []
    reserved_rows = []

    for subject_id, runs in subject_map.items():
        print(f"\n=== {subject_id} ===")

        reserved_ictal_run, reserved_nonictal_run = (None, None)
        if subject_id in LLM_COHORT_SUBJECTS:
            reserved_ictal_run, reserved_nonictal_run = find_reserved_runs(runs, subject_id)
            print(f"[LLM reserved ictal] {reserved_ictal_run}")

        runs_for_internal = [r for r in runs if r.run_stem != reserved_ictal_run]

        if len(runs_for_internal) < 2:
            print("[SKIP internal] not enough runs")
            continue

        # leave-one-run-out internal CV
        for fold_id, test_run in enumerate(runs_for_internal):
            remaining = [r for r in runs_for_internal if r.run_stem != test_run.run_stem]
            if len(remaining) < 1:
                continue

            val_run_stem = choose_validation_run(remaining, test_run.run_stem)
            train_runs = [r for r in remaining if r.run_stem != val_run_stem]
            val_runs = [r for r in remaining if r.run_stem == val_run_stem]

            if len(train_runs) < 1:
                continue

            train_data = concatenate_runs_with_subject_idx(train_runs, subject_to_idx)
            val_data = concatenate_runs_with_subject_idx(val_runs, subject_to_idx)
            test_data = concatenate_runs_with_subject_idx([test_run], subject_to_idx)

            result = train_one_fold_adv(
                train_data=train_data,
                val_data=val_data,
                test_data=test_data,
                subject_to_idx=subject_to_idx,
                device=DEVICE,
                adv_weight=ADV_WEIGHT,
                grl_lambda=GRL_LAMBDA,
            )

            m = result["test_metrics"]

            row = {
                "subject_id": subject_id,
                "fold_id": fold_id,
                "held_out_run": test_run.run_stem,
                "val_run": val_run_stem,
                "best_threshold": result["best_threshold"],
                "train_n_after_retention": result["train_n_after_retention"],
                "test_acc": m["acc"],
                "test_balanced_acc": m["balanced_acc"],
                "test_precision": m["precision"],
                "test_recall": m["recall"],
                "test_f1": m["f1"],
                "test_auroc": m["auroc"],
                "test_auprc": m["auprc"],
                "n_test": int(len(result["test_y"])),
                "n_test_ictal": int((result["test_y"] == 1).sum()),
                "n_test_nonictal": int((result["test_y"] == 0).sum()),
                "tn": m["tn"],
                "fp": m["fp"],
                "fn": m["fn"],
                "tp": m["tp"],
            }
            internal_rows.append(row)

            artifact_dir = save_fold_artifacts(
                subject_id=subject_id,
                fold_name=f"fold_{fold_id}",
                result=result,
                model_config=model_config,
                train_runs=[r.run_stem for r in train_runs],
                val_runs=[r.run_stem for r in val_runs],
                test_runs=[test_run.run_stem],
            )

            print(
                f"[fold {fold_id}] {test_run.run_stem} | "
                f"bal_acc={m['balanced_acc']:.3f} f1={m['f1']:.3f} "
                f"thr={result['best_threshold']:.2f} | saved -> {artifact_dir}"
            )

        # reserved evaluation
        if reserved_ictal_run:
            train_pool = [r for r in runs if r.run_stem != reserved_ictal_run]
            if len(train_pool) >= 2:
                val_run_stem = choose_validation_run(train_pool, reserved_ictal_run)
                train_runs = [r for r in train_pool if r.run_stem != val_run_stem]
                val_runs = [r for r in train_pool if r.run_stem == val_run_stem]
                test_runs = [r for r in runs if r.run_stem == reserved_ictal_run]

                train_data = concatenate_runs_with_subject_idx(train_runs, subject_to_idx)
                val_data = concatenate_runs_with_subject_idx(val_runs, subject_to_idx)
                test_data = concatenate_runs_with_subject_idx(test_runs, subject_to_idx)

                result = train_one_fold_adv(
                    train_data=train_data,
                    val_data=val_data,
                    test_data=test_data,
                    subject_to_idx=subject_to_idx,
                    device=DEVICE,
                    adv_weight=ADV_WEIGHT,
                    grl_lambda=GRL_LAMBDA,
                )

                m = result["test_metrics"]

                reserved_rows.append({
                    "subject_id": subject_id,
                    "reserved_ictal_run": reserved_ictal_run,
                    "reserved_interictal_run": reserved_nonictal_run,
                    "reserved_threshold_used": result["best_threshold"],
                    "reserved_acc": m["acc"],
                    "reserved_balanced_acc": m["balanced_acc"],
                    "reserved_precision": m["precision"],
                    "reserved_recall": m["recall"],
                    "reserved_f1": m["f1"],
                    "reserved_auroc": m["auroc"],
                    "reserved_auprc": m["auprc"],
                    "reserved_n_test": int(len(result["test_y"])),
                    "reserved_n_ictal": int((result["test_y"] == 1).sum()),
                    "reserved_n_nonictal": int((result["test_y"] == 0).sum()),
                    "tn": m["tn"],
                    "fp": m["fp"],
                    "fn": m["fn"],
                    "tp": m["tp"],
                })

                artifact_dir = save_fold_artifacts(
                    subject_id=subject_id,
                    fold_name="reserved",
                    result=result,
                    model_config=model_config,
                    train_runs=[r.run_stem for r in train_runs],
                    val_runs=[r.run_stem for r in val_runs],
                    test_runs=[reserved_ictal_run],
                )

                print(
                    f"[reserved] {reserved_ictal_run} | "
                    f"bal_acc={m['balanced_acc']:.3f} f1={m['f1']:.3f} "
                    f"thr={result['best_threshold']:.2f} | saved -> {artifact_dir}"
                )

    # save summary CSVs
    internal_df = pd.DataFrame(internal_rows)
    reserved_df = pd.DataFrame(reserved_rows)

    internal_path = EXPERIMENT_ROOT / "all_internal_cv_results.csv"
    reserved_path = EXPERIMENT_ROOT / "all_llm_reserved_results.csv"

    internal_df.to_csv(internal_path, index=False)
    reserved_df.to_csv(reserved_path, index=False)

    print("\nSaved:")
    print(internal_path)
    print(reserved_path)

    if len(internal_df) > 0:
        summary = (
            internal_df.groupby("subject_id")
            .agg(
                n_runs=("held_out_run", "count"),
                mean_test_accuracy=("test_acc", "mean"),
                mean_test_balanced_accuracy=("test_balanced_acc", "mean"),
                mean_test_precision=("test_precision", "mean"),
                mean_test_recall=("test_recall", "mean"),
                mean_test_f1=("test_f1", "mean"),
                mean_test_auroc=("test_auroc", "mean"),
                mean_test_auprc=("test_auprc", "mean"),
                std_test_f1=("test_f1", "std"),
                std_test_recall=("test_recall", "std"),
            )
            .reset_index()
        )
        summary_path = EXPERIMENT_ROOT / "cross_subject_summary.csv"
        summary.to_csv(summary_path, index=False)
        print(summary_path)


if __name__ == "__main__":
    main()


=== sub-HUP060 ===


C:\Users\ajars\anaconda3\Lib\site-packages\torch\nn\modules\transformer.py:286: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")
C:\Users\ajars\anaconda3\Lib\site-packages\torch\nn\functional.py:5476: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:263.)
  attn_output = scaled_dot_product_attention(q, k, v, attn_mask, dropout_p, is_causal)


[fold 0] sub-HUP060_ses-presurgery_task-ictal_acq-seeg_run-01 | bal_acc=0.875 f1=0.939 thr=0.35 | saved -> D:\hup_all_subjects_adv_transformer\models\sub-HUP060\fold_0


C:\Users\ajars\anaconda3\Lib\site-packages\torch\nn\modules\transformer.py:286: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


[fold 1] sub-HUP060_ses-presurgery_task-ictal_acq-seeg_run-02 | bal_acc=0.919 f1=0.915 thr=0.55 | saved -> D:\hup_all_subjects_adv_transformer\models\sub-HUP060\fold_1


...

[fold 0] sub-HUP190_ses-presurgery_task-ictal_acq-seeg_run-01 | bal_acc=0.986 f1=0.987 thr=0.20 | saved -> D:\hup_all_subjects_adv_transformer\models\sub-HUP190\fold_0


C:\Users\ajars\anaconda3\Lib\site-packages\torch\nn\modules\transformer.py:286: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


[fold 1] sub-HUP190_ses-presurgery_task-ictal_acq-seeg_run-02 | bal_acc=0.898 f1=0.905 thr=0.30 | saved -> D:\hup_all_subjects_adv_transformer\models\sub-HUP190\fold_1


C:\Users\ajars\anaconda3\Lib\site-packages\torch\nn\modules\transformer.py:286: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


[fold 2] sub-HUP190_ses-presurgery_task-ictal_acq-seeg_run-03 | bal_acc=0.910 f1=0.899 thr=0.40 | saved -> D:\hup_all_subjects_adv_transformer\models\sub-HUP190\fold_2

Saved:
D:\hup_all_subjects_adv_transformer\all_internal_cv_results.csv
D:\hup_all_subjects_adv_transformer\all_llm_reserved_results.csv
D:\hup_all_subjects_adv_transformer\cross_subject_summary.csv
